In [ ]:
import gdsfactory as gf
gf.gpdk.PDK.activate()
from gdsfactory.component import Component
from gdsfactory.cross_section   import CrossSection
from gdsfactory.typings import ComponentSpec, CrossSectionSpec, Position, LayerSpec, Port
from cross_sections import xs_ekn300_te_IMGREV
from typing import Any

port_names_electrical: gf.typings.IOPorts = ("e1", "e2")
port_types_electrical: gf.typings.IOPorts = ("electrical", "electrical")

In [2]:
@gf.xsection
def heater_metal_trench(
    width: float = 2.5,
    layer: gf.typings.LayerSpec = "HEATER",
    layer_trench: gf.typings.LayerSpec = (2,0),
    radius: float | None = None,
    port_names: gf.typings.IOPorts = port_names_electrical,
    port_types: gf.typings.IOPorts = port_types_electrical,
    width_trench: float = 1.0,
    offset: float = 0.0,
    **kwargs: Any,
) -> CrossSection:
    trench_center = (width + width_trench) / 2

    sections = (
        # gf.Section(
        #     width=width,
        #     offset=0,
        #     layer=layer_trench,
        #     name="_default",
        # ),

        gf.Section(
            width=width_trench*2+width,
            offset=0,
            layer=layer_trench,
            name="trench_top",
        ),
        # gf.Section(
        #     width=width_trench,
        #     offset=offset - trench_center,
        #     layer=layer_trench,
        #     name="trench_bot",
        # ),
    )

    """Return Metal Strip cross_section."""
    radius = radius or width
    return gf.cross_section.cross_section(
        width=width,
        layer=layer,
        radius=radius,
        port_names=port_names,
        port_types=port_types,
        sections=sections,
        **kwargs,
    )


In [3]:
@gf.cell_with_module_name
def taper_test(
    length: float = 10.0,
    width1: float = 0.5,
    width2: float | None = None,
    layer: LayerSpec | None = None,
    port: Port | None = None,
    with_two_ports: bool = True,
    cross_section: CrossSectionSpec = "strip",
    port_names: tuple[str, str] = ("o1", "o2"),
    port_types: tuple[str, str] = ("optical", "optical"),
    with_bbox: bool = True,
) -> Component:
    """Linear taper, which tapers only the main cross section section.

    Args:
        length: taper length.
        width1: width of the west/left port.
        width2: width of the east/right port. Defaults to width1.
        layer: layer for the taper.
        port: can taper from a port instead of defining width1.
        with_two_ports: includes a second port.
            False for terminator and edge coupler fiber interface.
        cross_section: specification (CrossSection, string, CrossSectionFactory dict).
        port_names: input and output port names. Second name only used if with_two_ports.
        port_types: input and output port types. Second type only used if with_two_ports.
        with_bbox: box in bbox_layers and bbox_offsets to avoid DRC sharp edges.
    """
    if len(port_types) != 2:
        raise ValueError("port_types should have two elements")

    x1 = gf.get_cross_section(cross_section, width=width1)
    if width2:
        width2 = gf.snap.snap_to_grid2x(width2)
        x2 = gf.get_cross_section(cross_section, width=width2)
    else:
        x2 = x1

    width1 = x1.width
    width2 = x2.width
    width_max = max([width1, width2])
    if layer:
        x = gf.get_cross_section(cross_section, width=width_max, layer=layer)
    else:
        x = gf.get_cross_section(cross_section, width=width_max)
    layer = layer or x.layer
    assert layer is not None

    if isinstance(port, gf.Port):
        width1 = port.width

    width2 = width2 or width1
    c = gf.Component()
    y1 = width1 / 2
    y2 = width2 / 2

    if length:
        p1 = gf.kdb.DPolygon(
            [
                gf.kdb.DPoint(0, y1),
                gf.kdb.DPoint(length, y2),
                gf.kdb.DPoint(length, -y2),
                gf.kdb.DPoint(0, -y1),
            ]
        )
        c.add_polygon(p1, layer=layer)

        s0_width = x.sections[0].width

        print(s0_width)

        for section in x.sections[1:]:
            delta_width = abs(section.width - s0_width)
            y1 = (width1 + delta_width) / 2
            y2 = (width2 + delta_width) / 2
            offset = section.offset
            p1 = gf.kdb.DPolygon(
                [
                    gf.kdb.DPoint(0, offset + y1),
                    gf.kdb.DPoint(length, offset + y2),
                    gf.kdb.DPoint(length, offset - y2),
                    gf.kdb.DPoint(0, offset - y1),
                ]
            )
            c.add_polygon(p1, layer=section.layer)


    if with_bbox:
        x.add_bbox(c)
    c.add_port(
        name=port_names[0],
        center=(0, 0),
        width=width1,
        orientation=180,
        layer=layer,
        cross_section=x1,
        port_type=port_types[0],
    )
    if with_two_ports:
        c.add_port(
            name=port_names[1],
            center=(length, 0),
            width=width2,
            orientation=0,
            layer=layer,
            cross_section=x2,
            port_type=port_types[1],
        )

    c.info["length"] = length
    c.info["width1"] = float(width1)
    c.info["width2"] = float(width2)
    return c


In [4]:
taper_test(cross_section=heater_metal_trench, width1=10, width2=1).show()

10.0
